# 02｜讀懂 agent 寫的程式：型態、JSON 與測試

**115-1 地理資訊系統運用程式｜W2（09/24）**

這份 notebook 分成兩部分：

- **Part A 讀程式（不需要網路）：** 型態、真假值、list 與 dict、`for` 迴圈、函式、traceback。
- **Part B JSON 與 API：** 用 `requests` 取得臺北市 YouBike 2.0 即時資料，檢查欄位，存成快照。

> **本週的重點不是「寫出程式」，而是「看懂 agent 寫的程式在做什麼，並且驗證它對不對」。**

標示 **【練習】** 的儲存格請自己動手完成。

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 0. 路徑設定

沿用 `00_Colab環境設定範本.ipynb` 的路徑設定。之後存檔都用 `PROCESSED_DIR` 組合路徑，不要寫死路徑。

In [3]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/115-1_GISprog")
else:
    # 備援：本機執行時，notebook 位於 115-1_GISprog/notebooks/，上一層就是專案資料夾
    PROJECT_DIR = Path.cwd().parent

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR = PROJECT_DIR / "outputs"

if not RAW_DIR.exists():
    raise FileNotFoundError(f"找不到 {RAW_DIR}，請確認 115-1_GISprog 資料夾位置是否正確")

print("執行環境：", "Google Colab" if IN_COLAB else "本機")
print("專案資料夾：", PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
執行環境： Google Colab
專案資料夾： /content/drive/MyDrive/115-1_GISprog


---

# Part A｜讀程式

## A1. 型態：先問「這是什麼型態？」

看 agent 寫的程式時，第一個要問的問題是：**這個變數是什麼型態？**

型態錯了，程式不一定會出錯，但結果會錯——這種錯最難發現。

In [4]:
值們 = [1800, "1800", 1800.0, True, None, ["a", "b"], {"sno": "500101001"}]

for v in 值們:
    print(f"{repr(v):<25} 型態 = {type(v).__name__}")

#　repr(v)：取得物件的「官方字串表示法」。
# 這非常關鍵，因為如果只印出 v，字串 "1800" 和整數 1800 在畫面上看起來會一模一樣。
# 使用 repr() 會保留字串的引號（如 '1800'），讓你能清楚看出它是字串。
# :<25：這是排版指令，表示將前面的內容向左對齊（<）並佔用 25 個字元寬度。
# 這確保了後面的「型態 = ...」能整齊地排在同一行，形成漂亮的表格感。
# type(v) 會回傳型態物件（例如 <class 'int'>）。
# 加上 .__name__ 可以只萃取出乾淨的型態名稱字串（例如 'int'）。

1800                      型態 = int
'1800'                    型態 = str
1800.0                    型態 = float
True                      型態 = bool
None                      型態 = NoneType
['a', 'b']                型態 = list
{'sno': '500101001'}      型態 = dict


In [5]:
# 字串和數字不一樣：看起來像數字的字串，不能拿來計算
a = "10"
b = "3"

print("字串相加：", a + b)          # 接在一起，不是 13
print("轉成整數再相加：", int(a) + int(b))
print("字串比大小：", "10" < "3")     # 逐字比較，所以 "10" 比 "3" 小
print("數字比大小：", 10 < 3)

字串相加： 103
轉成整數再相加： 13
字串比大小： True
數字比大小： False


## A2. 真假值：**這一頁是本週最容易踩到的陷阱**

`if x:` 並不是在問「x 是不是 True」，而是在問「x 是不是**有內容**」。

Python 認定為「假」的只有這幾種：`False`、`0`、`0.0`、`""`（空字串）、`[]`、`{}`、`None`。

**其他全部都是真——包含字串 `"0"`。**

**意即`if x:` 的意思等同於 `if bool(x):`。**

In [6]:
測試值 = [1, 0, "1", "0", "", "False", [], [0], {}, None]

for v in 測試值:
    print(f"{repr(v):<10} → if 判斷結果 = {bool(v)}")

1          → if 判斷結果 = True
0          → if 判斷結果 = False
'1'        → if 判斷結果 = True
'0'        → if 判斷結果 = True
''         → if 判斷結果 = False
'False'    → if 判斷結果 = True
[]         → if 判斷結果 = False
[0]        → if 判斷結果 = True
{}         → if 判斷結果 = False
None       → if 判斷結果 = False


**為什麼這件事很重要？**

YouBike 資料中，「這個站是否營運中」的欄位 `act` 是**字串** `"1"` (有營運)或 `"0"`(未營運)。

所以 agent 常寫的這一行是錯的：

```python
if station["act"]:      # "0" 也是 True，暫停營運的站會被算進去
    ...
```

正確的寫法是明確比較：

```python
if station["act"] == "1":
    ...
```

In [7]:
# 用兩筆手造資料驗證
站A = {"sna": "營運中的站", "act": "1"}
站B = {"sna": "暫停營運的站", "act": "0"}

for 站 in [站A, 站B]:
    print(f'{站["sna"]}：if act → {bool(站["act"])}，act == "1" → {站["act"] == "1"}')

營運中的站：if act → True，act == "1" → True
暫停營運的站：if act → True，act == "1" → False


## A3. list 與 dict：資料長什麼樣子

- **list `[...]`：** 有順序，用位置取值，例如 `records[0]`。
- **dict `{...}`：** 用鍵（key）取值，例如 `record["sarea"]`。

開放資料最常見的形狀就是 **list of dict**：一個 list，裡面每一筆是一個 dict。

In [8]:
records = [
    {"sno": "001", "sna": "第一站", "sarea": "大安區", "act": "1", "available_rent_bikes": 5, "Quantity": 20},
    {"sno": "002", "sna": "第二站", "sarea": "大安區", "act": "0", "available_rent_bikes": 0, "Quantity": 15},
    {"sno": "003", "sna": "第三站", "sarea": "中正區", "act": "1", "available_rent_bikes": 12, "Quantity": 30},
]

print("共幾筆：", len(records))
print("第一筆：", records[0])
print("第一筆有哪些 key：", list(records[0].keys()))
print("第一筆的區：", records[0]["sarea"])

共幾筆： 3
第一筆： {'sno': '001', 'sna': '第一站', 'sarea': '大安區', 'act': '1', 'available_rent_bikes': 5, 'Quantity': 20}
第一筆有哪些 key： ['sno', 'sna', 'sarea', 'act', 'available_rent_bikes', 'Quantity']
第一筆的區： 大安區


In [9]:
# 取不存在的 key 會出錯；用 .get() 可以給預設值
print(records[0].get("sarea", "（沒有這個欄位）"))
print(records[0].get("total", "（沒有這個欄位）"))
print(records[0])

大安區
（沒有這個欄位）
{'sno': '001', 'sna': '第一站', 'sarea': '大安區', 'act': '1', 'available_rent_bikes': 5, 'Quantity': 20}


## A4. `for` 迴圈累加：用 dict 分組統計

「各區有幾站」這種問題，標準做法是用一個 dict 當計數器。

In [10]:
站數 = {}                           # 定義一個空 dict 作為計數器

for r in records:
    區 = r["sarea"]
    站數[區] = 站數.get(區, 0) + 1   # 第一次遇到這個區時給 0，再加 1

print(站數)

{'大安區': 2, '中正區': 1}


**【練習 A4】** 把上面的程式改成只統計**營運中**的站數。

提示：在迴圈裡加一行判斷，注意 A2 的陷阱。

In [20]:
營運中站數 = {}

for r in records:
    # TODO：只有營運中的站才計入
    if (r["act"] == "1"):
        區 = r["sarea"]
        營運中站數[區] = 營運中站數.get(區, 0) + 1
print(營運中站數)
# 預期結果：{'大安區': 1, '中正區': 1}

{'大安區': 1, '中正區': 1}


## A5. 函式：讀懂輸入、輸出與 docstring

讀 agent 寫的函式時，先看三件事：

1. **參數**：需要什麼資料？
2. **回傳值**：吐出什麼？是 dict、list 還是 None？
3. **docstring**：它「宣稱」自己在做什麼——宣稱不等於事實，要驗證。

In [12]:
def count_by_area(records):
    """計算各行政區的站數。

    參數：
        records: list of dict，每筆至少要有 "sarea" 欄位。
    回傳：
        dict，key 是行政區，value 是站數。
    """
    result = {}
    for r in records:
        result[r["sarea"]] = result.get(r["sarea"], 0) + 1
    return result


print(count_by_area(records))

{'大安區': 2, '中正區': 1}


**Docstring在哪裡？**

```python
    """計算各行政區的站數。

    參數：
        records: list of dict，每筆至少要有 "sarea" 欄位。
    回傳：
        dict，key 是行政區，value 是站數。
    """
```
**Docstring 跟一般註解（#）有什麼不同？**

一般的 # 註解是寫給看原始碼的人看的，Python 執行時會完全忽略它。
而 Docstring 則會被 Python 系統真正儲存起來。如果你在程式其他地方輸入 help(count_by_area)，Python 就會直接把這段說明書印在畫面上給你參考。

In [21]:
help(count_by_area)

Help on function count_by_area in module __main__:

count_by_area(records)
    計算各行政區的站數。

    參數：
        records: list of dict，每筆至少要有 "sarea" 欄位。
    回傳：
        dict，key 是行政區，value 是站數。



## A6. `assert`：用自己手造的小資料驗證函式

不要用 1,800 筆真實資料去檢查函式對不對——**用你自己造的 3 筆資料**，因為你知道正確答案是什麼。

`assert 條件, "訊息"`：條件為真時什麼都不會發生；為假時丟出 `AssertionError`。

In [13]:
測試資料 = [
    {"sno": "t1", "sarea": "A區", "act": "1"},
    {"sno": "t2", "sarea": "A區", "act": "0"},
    {"sno": "t3", "sarea": "B區", "act": "1"},
]

結果 = count_by_area(測試資料)

assert 結果["A區"] == 2, f"A區應該有 2 站，實際是 {結果['A區']}"
assert 結果["B區"] == 1, f"B區應該有 1 站，實際是 {結果['B區']}"
assert len(結果) == 2, f"應該只有 2 個區，實際是 {len(結果)}"

print("3 個測試都通過")

3 個測試都通過


**【練習 A6】** 寫一個 `assert`，測試「暫停營運的站不應該被計入」。

這個測試現在**應該要失敗**，因為 `count_by_area` 根本沒有看 `act` 欄位。
先讓它失敗，看清楚錯誤訊息，這就是測試的價值。

In [24]:
# TODO：寫一個會失敗的 assert，證明 count_by_area 沒有處理 act
# 例如：測試資料的 A區 有 1 站營運中、1 站暫停，若函式正確應該回傳 1
assert count_by_area(測試資料)["A區"] == 1, f"暫停營運的站不應該被計入"

AssertionError: 暫停營運的站不應該被計入

## A7. 讀 traceback：由下往上看

程式出錯時，Python 會印出一長串紅字。讀法是：

1. **從最後一行開始看**：錯誤的「種類」與「說明」，例如 `KeyError: 'total'`。
2. **往上找你自己的程式碼那一行**：檔案路徑是你的 notebook、不是套件內部的那一行。
3. **再想「為什麼」**：`KeyError` 代表這個 key 不存在——是打錯字，還是資料根本沒有這個欄位？

下面這個儲存格會故意出錯，請執行它，讀懂錯誤訊息。

In [15]:
# 故意出錯：agent 以為有 total 欄位，但實際資料叫 Quantity
for r in records:
    print(r["sna"], r["total"])

KeyError: 'total'

**常見錯誤種類**

| 錯誤 | 意思 | 先檢查什麼 |
|---|---|---|
| `KeyError: 'total'` | dict 沒有這個 key | 印出 `list(record.keys())` 看真正的欄位名 |
| `TypeError` | 型態不對，例如字串和數字相加 | 印出 `type(變數)` |
| `IndexError` | list 索引超出範圍 | 印出 `len(list)` |
| `NameError` | 變數還沒定義 | 上面的儲存格是不是沒執行？ |
| `FileNotFoundError` | 路徑找不到檔案 | 印出路徑，確認是不是寫死了本機路徑 |

> **Colab 的儲存格可以跳著執行，但變數是共用的。** 出現 `NameError` 時，先確認上面的儲存格都執行過。

---

# Part B｜JSON 與 API

## B1. JSON 就是「文字版的 list 和 dict」

JSON 是資料在網路上傳輸時的**文字格式**。用 `json.loads()` 轉成 Python 物件之後，它就只是 list 和 dict。

| JSON | Python |
|---|---|
| `{...}` object | `dict` |
| `[...]` array | `list` |
| `"文字"` | `str` |
| `123` / `1.5` | `int` / `float` |
| `true` / `false` | `True` / `False` |
| `null` | `None` |

In [34]:
import json

json_文字 = '{"sno": "500101001", "sna": "捷運科技大樓站", "act": "1", "Quantity": 28}'

資料 = json.loads(json_文字)

print("轉換後的型態：", type(資料).__name__)
print("Quantity 的型態：", type(資料["Quantity"]).__name__)
print("act 的型態：", type(資料["act"]).__name__, "← 注意，是字串")

轉換後的型態： dict
Quantity 的型態： int
act 的型態： str ← 注意，是字串


## B2. 用 `requests` 取得即時資料

臺北市 YouBike 2.0 即時資料（免金鑰、公開）：

`https://tcgbusfs.blob.core.windows.net/dotapp/youbike/v2/youbike_immediate.json`

三個一定要寫的東西：

- **`timeout=10`：** 沒有它，網路卡住時程式會永遠等下去。
- **`raise_for_status()`：** 對方回傳 404、500 時主動丟出錯誤，不要把錯誤頁面當成資料。
- **`response.json()`：** 把回應內容轉成 Python 的 list／dict。

In [35]:
import requests

URL = "https://tcgbusfs.blob.core.windows.net/dotapp/youbike/v2/youbike_immediate.json"

response = requests.get(URL, timeout=10)
response.raise_for_status()
data = response.json()

print("HTTP 狀態碼：", response.status_code)
print("轉換後的型態：", type(data).__name__)
print("共幾筆：", len(data))

HTTP 狀態碼： 200
轉換後的型態： list
共幾筆： 1803


## B3. 拿到資料之後：**先檢查，再寫程式**

這是本週最重要的工作習慣。agent 常常憑「印象中的欄位名」寫程式——它記得的欄位名可能來自別的資料集。

**先印出第一筆的所有 key 和型態**，再決定要用哪些欄位。

In [36]:
第一筆 = data[0]

for key, value in 第一筆.items():
    print(f"{key:<25} {type(value).__name__:<6} {value}")

sno                       str    500101001
sna                       str    YouBike2.0_捷運科技大樓站
sarea                     str    大安區
mday                      str    2026-09-24 11:20:03
ar                        str    復興南路二段235號前
sareaen                   str    Daan Dist.
snaen                     str    YouBike2.0_MRT Technology Bldg. Sta.
aren                      str    No.235， Sec. 2， Fuxing S. Rd.
act                       str    1
srcUpdateTime             str    2026-09-24 11:20:52
updateTime                str    2026-09-24 11:20:52
infoTime                  str    2026-09-24 11:20:03
infoDate                  str    2026-09-24
Quantity                  int    28
available_rent_bikes      int    3
latitude                  float  25.02605
longitude                 float  121.5436
available_return_bikes    int    23


**看完上面的輸出，請回答（寫在你的筆記裡）：**

1. 總車柱數的欄位叫什麼名字？和你預期的一樣嗎？
2. `act` 是什麼型態？
3. 時間欄位有幾個？它們是什麼型態？彼此有什麼差別？

In [27]:
# 檢查 2：所有筆的欄位是不是都一樣？（不要只看第一筆就下結論）
欄位集合 = {tuple(sorted(r.keys())) for r in data}
print("不同的欄位組合數量：", len(欄位集合))

# 檢查 3：act 實際出現過哪些值
print("act 出現過的值：", {r["act"] for r in data})

不同的欄位組合數量： 1
act 出現過的值： {'0', '1'}


## B4. 分組統計：各行政區有幾站

用 A4 的寫法統計真實資料。

In [37]:
各區站數 = {}

for r in data:
    各區站數[r["sarea"]] = 各區站數.get(r["sarea"], 0) + 1

for 區, 數量 in sorted(各區站數.items(), key=lambda x: x[1], reverse=True):
    print(f"{區:<12} {數量:>4} 站")

print("\n共有幾個 sarea 類別：", len(各區站數))

內湖區           226 站
大安區           216 站
中山區           205 站
士林區           155 站
中正區           138 站
北投區           133 站
信義區           129 站
文山區           124 站
南港區           122 站
松山區           108 站
萬華區            98 站
大同區            84 站
臺大公館校區         65 站

共有幾個 sarea 類別： 13


**lambda x: x[1] 是一個匿名函數**

- lambda：告訴 Python「我現在要當場做一個臨時的函數，懶得取名字了」。

- x：代表輸入的資料，也就是 ('大安區', 15)。

- x[1]：代表回傳的結果，也就是告訴 Python「請用第 1 個位置的資料（15）去比大小」。

**【練習 B4】** 上面印出的 `sarea` 類別，每一個都是臺北市的行政區嗎？

如果有不是行政區的類別，你在 HW2 要怎麼處理它？把你的決定和理由寫進筆記。

## B5. 資料合不合理：可借 ＋ 可還 ＝ 總車柱數？

直覺上，「可借車輛數 ＋ 可還空位數」應該等於「總車柱數」。**先猜，再驗證。**

In [38]:
小於 = 0
等於 = 0
大於 = 0

for r in data:
    合計 = r["available_rent_bikes"] + r["available_return_bikes"]
    if 合計 < r["Quantity"]:
        小於 += 1
    elif 合計 == r["Quantity"]:
        等於 += 1
    else:
        大於 += 1

print(f"可借＋可還 <  總車柱數：{小於} 站")
print(f"可借＋可還 == 總車柱數：{等於} 站")
print(f"可借＋可還 >  總車柱數：{大於} 站")

可借＋可還 <  總車柱數：697 站
可借＋可還 == 總車柱數：1106 站
可借＋可還 >  總車柱數：0 站


In [31]:
# 差額最大的 5 站長什麼樣子？
差額最大 = sorted(data, key=lambda r: r["Quantity"] - r["available_rent_bikes"] - r["available_return_bikes"], reverse=True)[:5]

for r in 差額最大:
    差 = r["Quantity"] - r["available_rent_bikes"] - r["available_return_bikes"]
    print(f'{r["sna"]:<35} 總柱={r["Quantity"]:>3} 可借={r["available_rent_bikes"]:>3} 可還={r["available_return_bikes"]:>3} 差={差:>3} act={r["act"]}')

YouBike2.0_榮星花園                     總柱= 46 可借=  0 可還=  0 差= 46 act=0
YouBike2.0_捷運動物園站(2號出口)             總柱= 39 可借=  0 可還=  0 差= 39 act=0
YouBike2.0_林安泰古厝                    總柱= 36 可借=  0 可還=  0 差= 36 act=0
YouBike2.0_復華花園新城                   總柱= 32 可借=  0 可還=  0 差= 32 act=0
YouBike2.0_螢橋公園                     總柱= 29 可借=  0 可還=  0 差= 29 act=0


**【練習 B5】** 看看差額最大的那幾站的 `act` 值，你發現了什麼？

差額的意義是什麼？資料裡有沒有任何欄位可以直接告訴你答案？如果沒有，你要去哪裡查？

> 這一題沒有標準答案。**重點是：你不知道的事，不要自己編一個解釋。** 查臺北市資料大平臺的欄位說明，或在筆記裡誠實寫「尚未確認」。

## B6. 存成快照：HW3 會用到

即時資料**過了就沒了**。今天抓到的這份資料，明天就抓不回來了。

所以：**抓到就存檔**，檔名要帶時間。

In [39]:
from datetime import datetime

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 檔名用「取得資料的時間」，例如 youbike_20260924_1030.json
取得時間 = datetime.now().strftime("%Y%m%d_%H%M")
快照路徑 = PROCESSED_DIR / f"youbike_{取得時間}.json"

with open(快照路徑, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print("已存檔：", 快照路徑)
print("檔案大小：", round(快照路徑.stat().st_size / 1024 / 1024, 2), "MB")

已存檔： /content/drive/MyDrive/115-1_GISprog/data/processed/youbike_20260924_0322.json
檔案大小： 1.1 MB


> **為什麼存到 `data/processed/`？** `data/raw/` 是老師提供的原始資料，設定為唯讀。你自己抓的快照存到 `data/processed/`。
>
> **`ensure_ascii=False`：** 沒有它，中文會被存成 `\u5927\u5b89\u5340` 這種看不懂的樣子。

**【練習 B6】** 重新讀回剛才存的檔案，確認筆數和剛才一樣。

In [ ]:
with open(快照路徑, encoding="utf-8") as f:
    讀回來的資料 = json.load(f)

# TODO：寫一個 assert，確認讀回來的筆數和 data 一樣

---

## 本週回顧

| 你學到的 | 為什麼重要 |
|---|---|
| 先問「這是什麼型態」 | 型態錯了，程式不會報錯，但結果是錯的 |
| `if "0"` 是 True | YouBike 的 `act` 是字串，直接 `if` 會把停用站算進去 |
| 拿到資料先印 key | agent 憑印象寫欄位名，會出現 `KeyError` |
| traceback 由下往上讀 | 最後一行是錯誤種類，往上找自己的程式 |
| 用手造小資料寫 `assert` | 你知道正確答案，才能驗證函式 |
| 即時資料要存快照 | 過了就抓不回來，HW3 要用 |

**HW2（09/29 截止）：** 用純 Python（不用 Pandas）完成 `fetch_youbike()` 與 `summarize_by_area()`，並用 `assert` 測試。詳見 `HW2_作業說明`。

**記得：** 打開 AI 之前，先寫筆記的 Before AI 欄位。